# Python Best Practices & OOP: model solutions

Model solutions for the post-training exercises.

In [ ]:
import pandas as pd

## Q1) Get to know the data

In [ ]:
# Q1a)
buildings = pd.read_csv("buildings.csv")
print(buildings.head())

# Q1b)
print(buildings.shape)

# Q1c)
print(buildings["heating_system"].unique())

# Q1d)
print((buildings["heating_system"] == "Gas boiler").sum(), "gas boiler archetypes")

## Q2) A function with a clear contract

In [ ]:
# Q2a) - Q2c)
def load_buildings(path="buildings.csv", region=None):
    """
    Load the buildings archetype data, optionally for one region only.

    Parameters
    ----------
    path : str
        Path to the csv file. Defaults to "buildings.csv".
    region : str, optional
        If given, keep only rows for this region.

    Returns
    -------
    pandas.DataFrame
        The archetypes, filtered to `region` when one is supplied.
    """
    data = pd.read_csv(path)
    if region is not None:
        data = data.loc[data["region"] == region]
    return data


# Q2d)
south = load_buildings(region="South England")
print(south.shape[0], "archetypes in South England, out of", buildings.shape[0])

# The function is not pure: it reads a file, so its output depends on what's on
# disk, not only on its arguments. That's fine here, but it means a test can't
# just call it in isolation - it needs a file to read. Q7 and Q9 come back to
# this by separating the load from the calculation.

## Q3) Helpful errors

In [ ]:
CARBON_FACTORS = {
    "Gas boiler": 0.183,
    "Oil boiler": 0.246,
    "Biofuel boiler": 0.020,
    "ASHP": 0.000,
}


# Q3a) - the rough version
def heating_emissions(fuel_kwh, system="Gas boiler"):
    return fuel_kwh * CARBON_FACTORS[system]


# Q3b) - an unknown system raises a bare KeyError, which points at the
# dictionary rather than telling the user what they did wrong
heating_emissions(10000, "Coal")  ->  KeyError: 'Coal'

In [ ]:
# Q3c) - the same function, with errors a caller can act on
def heating_emissions(fuel_kwh, system="Gas boiler"):
    """Emissions in kgCO2 from `fuel_kwh` kWh burned by `system`."""
    try:
        return fuel_kwh * CARBON_FACTORS[system]
    except KeyError:
        raise ValueError(
            "'system' should be one of: " + ", ".join(CARBON_FACTORS)
        )
    except TypeError:
        raise TypeError("'fuel_kwh' should be numeric")

# Notice the use of the .join() method for strings which has not been
# covered in the courses until now. This can be used to populate a list
# of values or the keys of a dictionary into a string.

# Tests
print(heating_emissions(10000, "Gas boiler"))   # 1830.0

In [ ]:
# Expect a ValueError
heating_emissions(10000, "Coal")

In [ ]:
# Expect a TypeError
heating_emissions("lots", "Gas boiler")

## Q4) Code style and naming

In [ ]:
# Q4a) - the same calculation, readable
import pandas as pd

def mean_intensity(data):
    """Mean space-heating CO2 per square metre across the archetypes."""
    intensity = data["space_heating_co2"] / data["floor_area"]
    return intensity.mean()

print(mean_intensity(buildings))

# Q4b) - ruff format would, among other things, put a blank line before the
# function, add spaces around = and /, and re-indent the body to four spaces.
# It will NOT change the name of the function since this could risk breaking
# the functionality of your code (and is therefore left to a human to edit).

## Q5) Your first class

In [ ]:
# Q5a) - Q5c)
class Dwelling:
    def __init__(self, name, floor_area, heating_system, co2):
        self.name = name
        self.floor_area = floor_area
        self.heating_system = heating_system
        self.co2 = co2

    def is_low_carbon(self):
        return self.co2 == 0


# Q5b)
home = Dwelling("Detached, South England", 102.8, "Gas boiler", 872.01)
print(home.floor_area, home.heating_system)

# Q5c)
print(home.is_low_carbon())   # False - a gas boiler has direct emissions

## Q6) Methods that compute, and methods that change

In [ ]:
# Q6a) - Q6c) - the finished class
class Dwelling:
    def __init__(self, name, floor_area, heating_system, co2):
        self.name = name
        self.floor_area = floor_area
        self.heating_system = heating_system
        self.co2 = co2
        self.retrofitted = False

    def is_low_carbon(self):
        return self.co2 == 0

    def intensity(self):
        return self.co2 / self.floor_area

    def retrofit(self, new_system, new_co2):
        self.heating_system = new_system
        self.co2 = new_co2
        self.retrofitted = True


# Q6d)
home = Dwelling("Detached, South England", 102.8, "Gas boiler", 872.01)
print(round(home.intensity(), 2), "kgCO2 per m2")

# The round() function has not been introduced in our courses previously.
# It's often useful to wrap the output from calculations in the round()
# function for reporting purposes (e.g., only reporting to 2 decimal
# places).

home.retrofit("ASHP", 0.0)
print(home.is_low_carbon(), home.retrofitted)   # True True

# `retrofitted` is part of the object's public story - a caller may reasonably
# want to know it - so it reads fine without a leading underscore. The _ is for
# attributes that are purely internal bookkeeping

## Q7) A class that owns its data

In [ ]:
# Q7a) - Q7d)
class BuildingsDataset:
    """A buildings archetype dataset loaded from a csv file."""

    def __init__(self, filepath):
        self.filepath = filepath
        self.raw_data = None

    def read(self):
        """Read the csv at self.filepath into the raw_data attribute."""
        self.raw_data = pd.read_csv(self.filepath)

    def mean_intensity(self):
        """Mean space-heating CO2 per square metre, once the data is read."""
        intensity = self.raw_data["space_heating_co2"] / self.raw_data["floor_area"]
        return intensity.mean()


ds = BuildingsDataset("buildings.csv")
print(ds.raw_data)              # None, before reading
ds.read()
print(round(ds.mean_intensity(), 2))

# Declaring raw_data = None in the constructor means the attribute always
# exists, so `ds.raw_data` is None rather than an AttributeError if someone
# forgets to call read() first.

## Q8) Inheritance

In [ ]:
# Q8a)
class EmissionsSource:
    def __init__(self, name, year, emissions):
        self.name = name
        self.year = year
        self.emissions = emissions

    def cumulative_emissions(self, years):
        # Emissions if this year's rate held for `years` years
        return self.emissions * years

    def describe(self):
        return f"{self.name} emitted {self.emissions} MtCO2e in {self.year}."


# Q8b) - super() for the shared attributes, then a method only a Dwelling can have
class Dwelling(EmissionsSource):
    def __init__(self, name, year, emissions, floor_area, heating_system):
        super().__init__(name, year, emissions)
        self.floor_area = floor_area
        self.heating_system = heating_system

    def intensity(self):
        return self.emissions / self.floor_area

    # Q8c) - override describe(), reusing the parent version with super()
    def describe(self):
        return super().describe() + f" It is heated by a {self.heating_system}."


# Q8d) - a subclass that adds nothing, so it keeps the inherited describe()
class Sector(EmissionsSource):
    def __init__(self, name, year, emissions):
        super().__init__(name, year, emissions)


# Q8e)
scottish_bungalow = Dwelling("Scottish Bungalow", 2025, 4.32, 151.54, "Gas boiler")
aviation = Sector("Aviation", 2025, 30.1)

sources = [scottish_bungalow, aviation]
for source in sources:
    print(source.describe())
    print("over 5 years:", source.cumulative_emissions(5))

# aviation.intensity() raises AttributeError: intensity() lives on Dwelling, and
# a Sector has no floor_area to divide by. The shared behaviour sits on the
# superclass; behaviour that needs a subclass's own attributes stays with it.

## Q9) Tests

Tests are just functions that raise `AssertionError` when the code is wrong. In a notebook you can define and call them; in a project they live in a `test_*.py` file that `pytest` discovers on its own.

```python
# test_buildings.py
import pytest

CARBON_FACTORS = {
    "Gas boiler": 0.183,
    "Oil boiler": 0.246,
    "Biofuel boiler": 0.020,
    "ASHP": 0.000,
}


def heating_emissions(fuel_kwh, system="Gas boiler"):
    try:
        return fuel_kwh * CARBON_FACTORS[system]
    except KeyError:
        raise ValueError(
            "'system' should be one of: " + ", ".join(CARBON_FACTORS)
        )
    except TypeError:
        raise TypeError("'fuel_kwh' should be numeric")


def test_heating_emissions():
    assert round(heating_emissions(10000, "Gas boiler"), 1) == 1830.0


def test_unknown_system_raises():
    with pytest.raises(ValueError):
        heating_emissions(10000, "Coal")
```

Running `pytest` from the terminal collects both tests and reports `2 passed`. `pytest.raises` is the clean way to assert that a call *should* fail - the test passes precisely because the `ValueError` was raised, and would fail if it weren't.

In [ ]:
# Q9a) - Q9b), runnable here as a check before moving them into a file
import pytest

CARBON_FACTORS = {
    "Gas boiler": 0.183,
    "Oil boiler": 0.246,
    "Biofuel boiler": 0.020,
    "ASHP": 0.000,
}


def test_heating_emissions():
    assert round(heating_emissions(10000, "Gas boiler"), 1) == 1830.0


def test_unknown_system_raises():
    with pytest.raises(ValueError):
        heating_emissions(10000, "Coal")


test_heating_emissions()
test_unknown_system_raises()
print("both tests pass")

## Q10) Into a module, and reproducible

The answer here is two files rather than a cell.

`helpers.py` holds the generic class, the one that isn't tied to buildings:

```python
# helpers.py
class EmissionsSource:
    def __init__(self, name, year, emissions):
        self.name = name
        self.year = year
        self.emissions = emissions

    def cumulative_emissions(self, years):
        return self.emissions * years

    def describe(self):
        return f"{self.name} emitted {self.emissions} MtCO2e in {self.year}."
```

`buildings_tools.py` holds the buildings-specific code, and imports the parent class from helpers:

```python
# buildings_tools.py
import pandas as pd

from helpers import EmissionsSource


CARBON_FACTORS = {
    "Gas boiler": 0.183,
    "Oil boiler": 0.246,
    "Biofuel boiler": 0.020,
    "ASHP": 0.000,
}


def load_buildings(path="buildings.csv", region=None):
    """Load the buildings archetype data, optionally for one region only."""
    data = pd.read_csv(path)
    if region is not None:
        data = data.loc[data["region"] == region]
    return data


def heating_emissions(fuel_kwh, system="Gas boiler"):
    """Emissions in kgCO2 from `fuel_kwh` kWh burned by `system`."""
    try:
        return fuel_kwh * CARBON_FACTORS[system]
    except KeyError:
        raise ValueError("'system' should be one of: " + ", ".join(CARBON_FACTORS))
    except TypeError:
        raise TypeError("'fuel_kwh' should be numeric")


class Dwelling(EmissionsSource):
    def __init__(self, name, year, emissions, floor_area, heating_system):
        super().__init__(name, year, emissions)
        self.floor_area = floor_area
        self.heating_system = heating_system

    def intensity(self):
        return self.emissions / self.floor_area

    def describe(self):
        return super().describe() + f" It is heated by a {self.heating_system}."


if __name__ == "__main__":
    data = load_buildings()
    intensity = (data["space_heating_co2"] / data["floor_area"]).mean()
    print("mean intensity:", round(intensity, 2), "kgCO2 per m2")
```

On a practical level, `helpers.py` is free of any buildings knowledge, so another script (e.g., transport or industry) could import the same `EmissionsSource` without dragging any irrelevant information along.

`%run buildings_tools.py` prints the mean intensity, because running a script executes the `if __name__ == "__main__"` block. `import buildings_tools` prints nothing. It only runs whatever code is found above this block, and makes the objects (variables, functions, classes) available to whichever script or notebook is importing the code.

In [ ]:
# Q10c) - Q10d), once buildings_tools.py exists:
#
#     from buildings_tools import load_buildings, Dwelling
#     load_buildings(region="Wales").shape
#     Dwelling("Scottish Bungalow", 2025, 4.32, 151.54, "Gas boiler").describe()
#
#     %run buildings_tools.py     # prints the mean intensity
#     import buildings_tools      # prints nothing